# 02 – Labels sanity-check EDA

Verify the output of `src/generate_labels.py`.

In [ ]:
import pandas as pd

labels = pd.read_parquet('../data/processed/labels.parquet')
catalog = pd.read_parquet('../data/processed/events_catalog.parquet')
matches = pd.read_parquet('../data/processed/matches.parquet')

print('Labels shape :', labels.shape)
print('Catalog shape:', catalog.shape)

## 1. Row count matches expectation

In [ ]:
n_matches = labels['match_id'].nunique()
n_match_events = catalog[catalog['scope'] == 'match'].shape[0]
n_team_events = catalog[catalog['scope'] == 'team'].shape[0]
expected = n_matches * n_match_events + n_matches * n_team_events * 2
print(f'Actual: {len(labels):,} | Expected: {expected:,} | Match: {len(labels) == expected}')

## 2. Base rate table (P(outcome=1) per event)

In [ ]:
base_rates = (
    labels.dropna(subset=['outcome'])
    .groupby('event_id')['outcome']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='base_rate')
)
base_rates

## 3. Team-wins sanity: exactly one winner per match (excl no-results)

In [ ]:
win_labels = labels[labels['event_id'] == 'team_wins_match'].copy()
win_labels = win_labels.merge(matches[['match_id', 'result']], on='match_id', how='left')
win_labels_nr = win_labels[win_labels['result'] != 'no result']
sums = win_labels_nr.groupby('match_id')['outcome'].sum()
violations = sums[sums != 1]
print(f'Matches checked: {len(sums)} | Violations: {len(violations)}')
if len(violations):
    print(violations.head(10))

## 4. Historical spot-check — SRH 287 match (2024-04-15, match_id 1426268)

In [ ]:
srh = labels[labels['match_id'] == '1426268']
check_events = [
    'team_score_gte_160', 'team_score_gte_180', 'team_score_gte_200',
    'match_runs_gte_300', 'match_sixes_gte_15', 'any_fifty', 'any_century'
]
srh[srh['event_id'].isin(check_events)][['event_id', 'team', 'outcome']]

## 5. Historical spot-check — Gayle 17-six match (2013-04-23, match_id 598027)

In [ ]:
gayle = labels[labels['match_id'] == '598027']
check_events = ['match_sixes_gte_15', 'any_big_six_hitter', 'any_century']
gayle[gayle['event_id'].isin(check_events)][['event_id', 'team', 'outcome']]

## 6. NaN coverage — every no-result match should have all 40 labels as NaN

In [ ]:
no_result_ids = set(matches[matches['result'] == 'no result']['match_id'])
nr_labels = labels[labels['match_id'].isin(no_result_ids)]
na_per_match = nr_labels.groupby('match_id')['outcome'].apply(lambda s: s.isna().sum())
print('NaN counts per no-result match:')
print(na_per_match.value_counts().sort_index())
print('All matches have 40 NaNs:', (na_per_match == 40).all())